# A PostgreSQL resource takes exactly one of a pool and connection parameters

Both at once is refused when the resource is built.

Use the repository Python environment and run the cells in order. This notebook has its own setup; no previous notebook is required.

## 1. Import the implementation

In [ ]:
!pip install -q "aoa-action-machine[postgres]"

In [2]:
import asyncio

from aoa.action_machine.exceptions import ConnectionValidationError
from aoa.action_machine.resources.postgres import PostgresPool, PostgresStorage

## 2. Declare the model

In [3]:
class RecordingConnection:
    """Record commands; do not parse SQL or simulate database storage."""
    def __init__(self, number):
        """Give this connection a visible identity."""
        self.number = number
        self.commands = []
        self.busy = False
        self.closed = False
    async def execute(self, query, *params):
        """Reject concurrent use and record one command."""
        if self.busy or self.closed:
            raise RuntimeError("connection is busy or closed")
        self.busy = True
        try:
            await asyncio.sleep(0)
            self.commands.append(query)
            return "OK"
        finally:
            self.busy = False
    async def close(self):
        """Mark the physical connection closed."""
        self.closed = True

class RecordingPool:
    """Offer a fixed number of observable connections without a server."""
    def __init__(self, size=1):
        """Create the fixed capacity used by this example."""
        self.connections = [RecordingConnection(i + 1) for i in range(size)]
        self.free = asyncio.Queue()
        for connection in self.connections:
            self.free.put_nowait(connection)
        self.held = set()
        self.closed = False
    async def acquire(self, *, timeout=None):
        """Wait for an available connection with the supplied bound."""
        connection = await asyncio.wait_for(self.free.get(), timeout)
        self.held.add(connection)
        return connection
    async def release(self, connection):
        """Return a borrowed connection exactly once."""
        self.held.remove(connection)
        self.free.put_nowait(connection)
    async def close(self):
        """Close the pool only after all borrowers have returned."""
        assert not self.held
        for connection in self.connections:
            await connection.close()
        self.closed = True



class ExampleStorage(PostgresStorage):
    """A PostgreSQL resource that works through the example's pool."""


async def main():
    """Run this case and verify its observable result."""
    try:
        ExampleStorage(pool=PostgresPool(pool=RecordingPool()), connection_params={"dsn": "example"})
    except ConnectionValidationError as error:
        print(type(error).__name__)
    else:
        raise AssertionError("Invalid construction was accepted")

## 3. Run and check the observation

In [4]:
await main()

ConnectionValidationError


## 4. Explain the result

A PostgreSQL storage takes exactly one of `pool=` and `connection_params=`. The check happens during construction, before open and before any database access.